# 04 · CDSE Process API: a masked optical image

**Objective:** Request physical B04/B08 reflectance for a small Finland AOI, mask poor-quality classes, and calculate NDVI.

**Route:** Live API / CDSE OAuth client · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** three arrays/maps and request provenance. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/cdse-sentinel-hub.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Configure a small request

Create a CDSE Sentinel Hub OAuth client and set `CDSE_CLIENT_ID` / `CDSE_CLIENT_SECRET` locally. The example uses a 256 × 256 output for a 2 km-scale AOI; these are **output dimensions**, not a guarantee of native 10 m sampling. The process service mosaics eligible scenes within the interval.


In [ ]:
from examples.sentinel import access_token, process_payload, process
from examples.raster import ndvi

bbox = [24.55, 60.15, 24.57, 60.17]
start, end = "2024-08-01T00:00:00Z", "2024-08-31T23:59:59Z"
payload = process_payload("sentinel-2-l2a", bbox, start, end, max_cloud=20)
print({"collection": payload["input"]["data"][0]["type"], "bbox": bbox, "time_range": [start, end], "output_pixels": [256, 256]})


In [ ]:
# Read the request and account prerequisites before changing this switch.
RUN_LIVE = False
if not RUN_LIVE:
    raise RuntimeError("Live retrieval is disabled. Complete account setup, review quotas, then set RUN_LIVE=True.")


In [ ]:
token = access_token()
image, grid = process(token, payload)
red, nir, valid, scl = image
result = ndvi(red, nir, valid > 0)
assert np.isfinite(result).any(), "No valid pixels remain; select another interval."
pd.Series(result[np.isfinite(result)]).describe()


## 3. Compare reflectance and the derived index

Reflectance output is already calibrated by the service; do not apply SAFE digital-number offsets again. Tile-level maximum cloud coverage narrows selection; per-pixel SCL removes retained cloud/shadow/snow classes.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(axes, [red, nir, result], ["B04 reflectance", "B08 reflectance", "Masked NDVI"]):
    im = ax.imshow(data)
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
plt.show()
record_provenance("sentinel-2-process.json", service="CDSE Sentinel Hub", collection="sentinel-2-l2a", request=payload, grid=grid, valid_pixels=int(np.isfinite(result).sum()), units="BOA reflectance and dimensionless NDVI")


## Interpret and verify

SCL is a pragmatic quality mask with classification errors, especially at peatland boundaries. Output is a mosaic, not an explicitly selected catalogue scene. Replace the rectangular AOI with validated site polygons for scientific site summaries.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [CDSE Process API](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Process.html)
- [CDSE authentication](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Overview/Authentication.html)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
